# EdNet-KT1 knowledge-tracing benchmark on Google Colab

Runs the full thesis experiment (Chapter 4, E1 and E2) on a Colab GPU:

* **Data:** nested EdNet-KT1 cohorts (300 ⊂ 1,000 ⊂ 2,000 ⊂ 5,000 ⊂ 10,000 ⊂ 20,000 [⊂ 50,000] learners). Each is split 70/10/20 by learner.
* **Models:** DKT, SAKT and AKT × seeds 42/43/44. Hyperparameters come from `configs/model_configs.json`: 50 epochs, early stop after 10 epochs with no validation gain.
* **Evaluation:** test-split metrics, then hardening:
  * BKT with default and fitted parameters, and the logistic-regression baseline;
  * bootstrap 95% CIs and paired AUC differences.
* **Learning curve:** test AUC against cohort size.

Everything is written to Google Drive while it runs, so a disconnect only loses the run in progress.

### Before the first run (on your own machine)
1. Build the bundle (code and cohort CSVs; Colab never needs the 1.2 GB EdNet zip):
   ```bash
   .venv-kt/bin/python knowledge_tracing/make_colab_bundle.py              # tiers core + extended
   .venv-kt/bin/python knowledge_tracing/make_colab_bundle.py --sizes 300 1000 2000 5000 10000 20000 50000   # + max
   ```
2. Upload `data/colab/ednet_colab_bundle.zip` to Google Drive as **`MyDrive/ednet_kt1/ednet_colab_bundle.zip`**.
3. In Colab: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

### Tiers and expected time (free T4)

| Tier | Cohorts | GPU time (upper bound) |
|---|---|---|
| `core` | 300, 1k, 2k, 5k: the thesis grid. pyKT's own EdNet setup also samples 5,000 users. | ≤ 45 min |
| `extended` (default) | core + 10k, 20k | ≈ 3–4 h |
| `max` | extended + 50k | ≈ 8–9 h in total; expect two sessions, since it resumes |

**Where the times come from.** An AKT epoch measured 102 s on the 50k cohort on a local GTX 1070 Ti, and DKT 11 s. A T4 is about as fast; that is assumed, not measured. Epoch time grows with cohort size.

**Why these are upper bounds.** Each run is capped at 50 epochs and stops early after 10 epochs without a validation gain; on the 300 cohort the best epoch was 6–37. Hardening (1,000 bootstrap resamples) adds roughly 20–40 min of CPU time on the 50k cohort. The notebook prints the real time of every run.

**Why not all 784k EdNet users?** Only 41% of users have at least 20 answers (50,000 of 122,239 users sampled), so the whole pool is about 320k learners and about 29M answers. That is 6.4× the 50k cohort: roughly 11 min per AKT epoch, or about 25 h of GPU time for three seeds. A free session would not finish it. Cohorts are run smallest first, so the thesis grid is always complete before the large cohorts start.

### If the session disconnects
Reconnect, set the same `TIER`, and run all cells again. Prepared cohorts, finished (model, seed) runs and finished evaluations are reused (`RESUME=1`). Only the interrupted run starts again.

## 1. Settings

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/ednet_kt1"   # holds the bundle; all outputs go here too
TIER = "core"                                  # "core" | "extended" | "max"
TIERS = {
    "core":     [300, 1000, 2000, 5000],
    "extended": [300, 1000, 2000, 5000, 10000, 20000],
    "max":      [300, 1000, 2000, 5000, 10000, 20000, 50000],
}
SIZES  = TIERS[TIER]
SEEDS  = [42, 43, 44]
MODELS = ["dkt", "sakt", "akt"]
EPOCHS = None            # None = model_configs.json (50, early stop after 10 flat epochs); never lower it for thesis numbers
RUN_SMOKE_TEST = True    # 2-epoch end-to-end check on the 300 cohort, written to /content/smoke (not Drive)
EXPORT_PREDICTIONS = True  # per-target test predictions in the results zip (needed to rerun hardening locally)

## 2. GPU check

In [ ]:
import subprocess, torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then run all cells again.")
print(torch.cuda.get_device_name(0), "|", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 3. Mount Drive and unpack the bundle

The code goes to `/content/ednet`, which is rebuilt every session. Data and results live on Drive at `DRIVE_ROOT/data`, and `/content/ednet/data` is a link to that folder. Cohort CSVs are copied only if they are new or have changed.

In [ ]:
import hashlib, json, os, shutil, sys, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
ROOT, REPO = Path(DRIVE_ROOT), Path("/content/ednet")
DATA = ROOT / "data"
BUNDLE = ROOT / "ednet_colab_bundle.zip"
if not BUNDLE.exists():
    raise SystemExit(f"{BUNDLE} not found: build it locally with knowledge_tracing/make_colab_bundle.py and upload it")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

shutil.rmtree(REPO, ignore_errors=True)
REPO.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as zf:
    BUNDLE_MANIFEST = json.loads(zf.read("BUNDLE_MANIFEST.json"))
    for name in zf.namelist():
        if name.startswith("data/"):
            dest = DATA / name[len("data/"):]
            if dest.exists() and sha256(dest) == BUNDLE_MANIFEST["sha256"][name]:
                continue
            print("copy ->", dest)
            dest.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(name) as src, open(dest, "wb") as out:
                shutil.copyfileobj(src, out)
        else:
            zf.extract(name, REPO)
(REPO / "data").symlink_to(DATA, target_is_directory=True)
os.chmod(REPO / "knowledge_tracing/run_ednet_benchmark.sh", 0o755)
(DATA / "benchmarks").mkdir(parents=True, exist_ok=True)
(ROOT / "logs").mkdir(exist_ok=True)
os.chdir(REPO)

print("bundle created:", BUNDLE_MANIFEST["created_utc"], "| git HEAD:", BUNDLE_MANIFEST["git_head"][:10] or "?")
if BUNDLE_MANIFEST["uncommitted_changes"]:
    print("bundled code includes uncommitted changes:", *BUNDLE_MANIFEST["uncommitted_changes"], sep="\n  ")

EDNET = DATA / "external" / "ednet"
missing = [n for n in SIZES if not (EDNET / f"kt1_events_{n}.csv").exists()]
if missing:
    raw = EDNET / "EdNet-KT1.zip"
    if not raw.exists():
        raise SystemExit(f"Cohorts {missing} are not in the bundle. Rebuild it locally with "
                         f"--sizes {' '.join(map(str, SIZES))}, or put EdNet-KT1.zip in {EDNET}.")
    # Fallback: build from the raw archive (same seed and filters, so the cohorts stay nested).
    local_zip = Path("/content/EdNet-KT1.zip")
    if not local_zip.exists():
        shutil.copy(raw, local_zip)   # random access into a zip on Drive is slow
    subprocess.run([sys.executable, "knowledge_tracing/datasets/ednet_kt1.py", "--kt1-zip", str(local_zip),
                    "--questions", str(EDNET / "questions.csv"), "--output-dir", str(EDNET),
                    "--sizes", *map(str, SIZES)], check=True)
print("cohorts ready:", SIZES)

## 4. Install pyKT

`--no-deps` keeps Colab's own torch/CUDA build: pyKT's model code only needs torch, numpy, pandas and scikit-learn, which Colab already has.

In [ ]:
!pip install -q --no-deps pykt-toolkit==0.0.38
from importlib import metadata
import platform, numpy, pandas, sklearn
from pykt.models import init_model, train_model, evaluate  # import check
ENV = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
       "gpu": torch.cuda.get_device_name(0), "numpy": numpy.__version__, "pandas": pandas.__version__,
       "scikit-learn": sklearn.__version__, "pykt-toolkit": metadata.version("pykt-toolkit"),
       "bundle_git_head": BUNDLE_MANIFEST["git_head"], "bundle_created_utc": BUNDLE_MANIFEST["created_utc"]}
(DATA / "benchmarks" / "colab_environment.json").write_text(json.dumps(ENV, indent=2))
ENV

## 5. Helpers: run with a live log, status table

In [ ]:
import time
import pandas as pd
from datetime import datetime
from IPython.display import Markdown, Image, display

LOG = ROOT / "logs" / f"run_{datetime.now():%Y%m%d_%H%M%S}.log"

def run_benchmark(sizes, bench_dir=None, epochs=EPOCHS, seeds=SEEDS, resume=True):
    """knowledge_tracing/run_ednet_benchmark.sh, output streamed here and appended to LOG on Drive."""
    env = dict(os.environ, PY=sys.executable, SIZES=" ".join(map(str, sizes)),
               SEEDS=" ".join(map(str, seeds)), MODELS=" ".join(MODELS), RESUME="1" if resume else "0")
    if epochs:
        env["EPOCHS"] = str(epochs)
    if bench_dir:
        env["BENCH_DIR"] = str(bench_dir)
    t0 = time.time()
    with open(LOG, "a") as log:
        proc = subprocess.Popen(["bash", "knowledge_tracing/run_ednet_benchmark.sh"], env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="")
            log.write(line); log.flush()
        rc = proc.wait()
    print(f"--- {sizes}: exit {rc}, {(time.time() - t0) / 60:.1f} min (log: {LOG})")
    if rc:
        raise RuntimeError(f"run_ednet_benchmark.sh failed for {sizes}; see the output above and {LOG}")

def status(bench=DATA / "benchmarks"):
    rows = []
    for n in SIZES:
        kt = bench / f"ednet_kt1_{n}" / "kt"
        ev = kt / "artifacts" / "evaluation"
        for m in MODELS:
            for s in SEEDS:
                f = kt / "artifacts" / m / f"seed{s}" / "run_summary.json"
                r = json.loads(f.read_text()) if f.exists() else {}
                rows.append({"cohort": n, "model": m.upper(), "seed": s, "trained": bool(r),
                             "epochs_cfg": r.get("training", {}).get("num_epochs"),
                             "best_epoch": r.get("best_epoch"), "valid_auc": r.get("valid_auc"),
                             "train_min": round(r["train_seconds"] / 60, 1) if r else None,
                             "evaluated": (ev / "test_comparison.json").exists(),
                             "hardened": (ev / "hardening_summary.json").exists()})
    return pd.DataFrame(rows)

status().pivot_table(index="cohort", columns="model", values="trained", aggfunc="sum")

## 6. Smoke test (about 2 minutes)

One seed for two epochs on the 300 cohort, run through every stage, written to `/content/smoke`. It checks the environment before hours of GPU time are spent. The numbers it prints are meaningless.

In [ ]:
if RUN_SMOKE_TEST:
    shutil.rmtree("/content/smoke", ignore_errors=True)
    run_benchmark([300], bench_dir="/content/smoke", epochs=2, seeds=[42], resume=False)

## 7. Full run

Cohorts run smallest first, each followed by its own evaluation, so the results below fill in while later cohorts are still training. Rerunning this cell after a disconnect resumes the run.

In [ ]:
T0 = time.time()
for n in sorted(SIZES):
    run_benchmark([n])
    s = status()
    s = s[s.cohort == n]
    print(f"cohort {n}: {int(s.trained.sum())}/{len(s)} runs trained | total so far {(time.time() - T0) / 3600:.2f} h")
status()

## 8. Results per cohort

In [ ]:
BENCH = DATA / "benchmarks"
for n in sorted(SIZES):
    ev = BENCH / f"ednet_kt1_{n}" / "kt" / "artifacts" / "evaluation"
    if not (ev / "test_comparison.md").exists():
        print(f"cohort {n}: not evaluated yet")
        continue
    display(Markdown(f"---\n# Cohort {n:,}\n\n" + (ev / "test_comparison.md").read_text()))
    hard = ev / "hardening_report.md"
    if hard.exists():   # the reliability tables are in the file on Drive; shown here: metrics, CIs, pairs
        display(Markdown(hard.read_text().split("## Reliability tables")[0]))

## 9. Learning curve (E2)

In [ ]:
curve = BENCH / "ednet_kt1_learning_curve.csv"
if curve.exists():
    df = pd.read_csv(curve)
    display(df.pivot_table(index="model", columns="cohort", values="auc_mean").round(4))
    if curve.with_suffix(".png").exists():
        display(Image(str(curve.with_suffix(".png"))))
else:
    print("No learning curve yet: run section 7 first.")

## 10. Export results

This writes `DRIVE_ROOT/ednet_kt1_results_<time>.zip`, which holds:
* evaluation reports, run summaries, training logs and run settings;
* split assignments and KC/question maps;
* the learning curve, the environment and the bundle manifest;
* per-target test predictions, if `EXPORT_PREDICTIONS` is on.

Checkpoints and pyKT input files stay on Drive and are left out. On your machine, unpack it next to (not over) the local runs:
```bash
unzip -o ~/Downloads/ednet_kt1_results_*.zip -d data/colab/
.venv-kt/bin/python knowledge_tracing/learning_curve.py --bench-dir data/colab/ednet_kt1_results/benchmarks
```

In [ ]:
SKIP_SUFFIX = {".ckpt", ".pth", ".pkl"}
out = ROOT / f"ednet_kt1_results_{datetime.now():%Y%m%d_%H%M}.zip"
n_files = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for n in sorted(SIZES):
        kt = BENCH / f"ednet_kt1_{n}" / "kt"
        if not kt.exists():
            continue
        picks = [kt / "data_config.json", *(kt / "processed").glob("*.json"),
                 *[kt / "processed" / f for f in ("split_assignments.csv", "question_map.csv", "concept_map.csv")],
                 *(p for p in (kt / "artifacts").rglob("*") if p.is_file())]
        for p in picks:
            if not p.exists() or p.suffix in SKIP_SUFFIX:
                continue
            if not EXPORT_PREDICTIONS and "predictions" in p.parts:
                continue
            zf.write(p, "ednet_kt1_results/benchmarks/" + str(p.relative_to(BENCH))); n_files += 1
    for p in [*BENCH.glob("ednet_kt1_learning_curve.*"), BENCH / "colab_environment.json"]:
        if p.exists():
            zf.write(p, f"ednet_kt1_results/benchmarks/{p.name}"); n_files += 1
    zf.writestr("ednet_kt1_results/BUNDLE_MANIFEST.json", json.dumps(BUNDLE_MANIFEST, indent=2))
    for p in (ROOT / "logs").glob("*.log"):
        zf.write(p, f"ednet_kt1_results/logs/{p.name}"); n_files += 1
print(f"{out} ({out.stat().st_size / 1e6:.1f} MB, {n_files} files)")
# from google.colab import files; files.download(str(out))   # or download it from Drive